In [9]:
import duckdb

con = duckdb.connect()
con.execute("PRAGMA threads=4")

summary_52 = con.execute("""
WITH period_data AS (
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        CAST(DATE AS DATE) AS period_start,
        SUM(CAST(COALESCE(ABVERKAUFTE_MENGE, 0) AS DOUBLE)) AS demand
    FROM read_parquet('../../data/processed/transactions_dst_over_weeks/*.parquet')
    GROUP BY ARTIKEL_ID, MARKT_ID, period_start
), series AS (
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        COUNT(*) AS active_weeks,
        SUM(CASE WHEN demand > 0 THEN 1 ELSE 0 END) AS demand_weeks,
        AVG(CASE WHEN demand > 0 THEN demand END) AS mean_nonzero_demand,
        VAR_SAMP(CASE WHEN demand > 0 THEN demand END) AS var_nonzero_demand
    FROM period_data
    GROUP BY ARTIKEL_ID, MARKT_ID
), metrics AS (
    SELECT
        *,
        active_weeks / NULLIF(demand_weeks, 0) AS ADI,
        CASE
            WHEN demand_weeks > 1 AND mean_nonzero_demand > 0
                THEN var_nonzero_demand / (mean_nonzero_demand * mean_nonzero_demand)
            WHEN demand_weeks = 1 THEN 0.0
            ELSE NULL
        END AS CV2
    FROM series
    WHERE demand_weeks > 0
), classified AS (
    SELECT
        *,
        CASE
            WHEN NOT isfinite(ADI) OR NOT isfinite(CV2) THEN 'unclassified'
            WHEN ADI < 1.32 AND CV2 < 0.49 THEN 'smooth'
            WHEN ADI < 1.32 AND CV2 >= 0.49 THEN 'erratic'
            WHEN ADI >= 1.32 AND CV2 < 0.49 THEN 'intermittent'
            ELSE 'lumpy'
        END AS demand_class
    FROM metrics
)
SELECT
    COUNT(*) AS total_series,
    COUNT(*) FILTER (WHERE demand_weeks >= 52) AS series_with_min_demand_weeks_52,
    COUNT(*) FILTER (WHERE demand_weeks >= 52 AND active_weeks >= 28) AS eval_eligible_with_52,
    COUNT(*) FILTER (WHERE demand_weeks >= 52 AND demand_class = 'smooth') AS smooth_52,
    COUNT(*) FILTER (WHERE demand_weeks >= 52 AND demand_class = 'erratic') AS erratic_52,
    COUNT(*) FILTER (WHERE demand_weeks >= 52 AND demand_class = 'intermittent') AS intermittent_52,
    COUNT(*) FILTER (WHERE demand_weeks >= 52 AND demand_class = 'lumpy') AS lumpy_52
FROM classified
""").fetchdf()

summary_52

   total_series  series_with_min_demand_weeks_52  eval_eligible_with_52  smooth_52  erratic_52  intermittent_52  lumpy_52
0        321529                            91988                  91988      29943       15369            17136     29540